In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType

STORAGE_ACCOUNT = "adlsairbnbde"

In [0]:
spark.conf.set(
    "fs.azure.account.key.adlsairbnbde.dfs.core.windows.net",
    "KEY HERE"
)
BRONZE_VALIDATED_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_validated/calendar"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/calendar"


In [0]:
df = spark.read.format("delta").load(BRONZE_VALIDATED_PATH)
print(f"Rows read from Bronze: {df.count()}")
display(df.groupBy("_source_city", "_source_quarter").count())

Rows read from Bronze: 31089612


_source_city,_source_quarter,count
lisbon,2026-Q2,9092881
lisbon,2025-Q3,9288887
barcelona,2025-Q3,7084654
barcelona,2026-Q2,5623190


In [0]:
if "price" in df.columns:
    non_null_price_by_quarter = (
        df.groupBy("_source_quarter")
          .agg(F.count("price").alias("non_null_price_rows"), F.count("*").alias("total_rows"))
    )
    display(non_null_price_by_quarter)
 
    total_non_null_price = df.filter(F.col("price").isNotNull()).count()
    print(f"Total non-null price rows across ALL quarters: {total_non_null_price}")

_source_quarter,non_null_price_rows,total_rows
2026-Q2,0,14716071
2025-Q3,0,16373541


Total non-null price rows across ALL quarters: 0


In [0]:
df = (
    df
    .withColumn("date_parsed", F.to_date("date"))
    .withColumn(
        "available_bool",
        F.when(F.col("available") == "t", True)
         .when(F.col("available") == "f", False)
         .otherwise(None)
    )
    .withColumn("minimum_nights_int", F.col("minimum_nights").cast(IntegerType()))
    .withColumn("maximum_nights_int", F.col("maximum_nights").cast(IntegerType()))
)

In [0]:

df = df.withColumn(
    "has_nights_anomaly",
    F.when(
        F.col("minimum_nights_int").isNotNull() &
        F.col("maximum_nights_int").isNotNull() &
        (F.col("minimum_nights_int") > F.col("maximum_nights_int")),
        True
    ).otherwise(False)
).withColumn(
    "has_valid_availability",
    F.col("available_bool").isNotNull()
)
 
anomaly_count = df.filter(F.col("has_nights_anomaly")).count()
invalid_availability_count = df.filter(~F.col("has_valid_availability")).count()
null_dates = df.filter(F.col("date_parsed").isNull()).count()
 
print(f"Rows flagged has_nights_anomaly=True: {anomaly_count}")
print(f"Rows with unparseable availability (has_valid_availability=False): {invalid_availability_count}")
print(f"Rows with unparseable date (should investigate if > 0): {null_dates}")

Rows flagged has_nights_anomaly=True: 2039
Rows with unparseable availability (has_valid_availability=False): 0
Rows with unparseable date (should investigate if > 0): 0


In [0]:
before_dedupe = df.count()
df = df.dropDuplicates(["listing_id", "date_parsed", "_source_city", "_source_quarter"])
after_dedupe = df.count()
print(f"Rows before dedupe: {before_dedupe}, after: {after_dedupe}, removed: {before_dedupe - after_dedupe}")

Rows before dedupe: 31089612, after: 31089612, removed: 0


In [0]:
silver_calendar = df.select(
    F.col("listing_id"),
    F.col("date_parsed").alias("date"),
    F.col("available_bool").alias("available"),
    F.col("has_valid_availability"),
    F.col("minimum_nights_int").alias("minimum_nights"),
    F.col("maximum_nights_int").alias("maximum_nights"),
    F.col("has_nights_anomaly"),
    F.col("_source_city").alias("city"),
    F.col("_source_quarter").alias("quarter_label"),
    F.col("_ingested_at"),
)
 
display(silver_calendar.limit(10))

listing_id,date,available,has_valid_availability,minimum_nights,maximum_nights,has_nights_anomaly,city,quarter_label,_ingested_at
1000085887158071609,2025-10-06,false,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-10-11,true,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-10-16,false,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-10-17,false,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-11-01,true,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-11-17,true,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-11-21,true,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-12-03,true,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-12-14,true,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069
1000085887158071609,2025-12-21,true,true,1,365,false,lisbon,2025-Q3,2026-07-27T11:07:36.285069


In [0]:
(
    silver_calendar.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("city", "quarter_label")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)
 
print(f"Silver calendar written to: {SILVER_PATH}")
print(f"Total rows: {silver_calendar.count()}")
display(silver_calendar.groupBy("city", "quarter_label").count())

Silver calendar written to: abfss://silver@adlsairbnbde.dfs.core.windows.net/calendar
Total rows: 31089612


city,quarter_label,count
barcelona,2025-Q3,7084654
barcelona,2026-Q2,5623190
lisbon,2025-Q3,9288887
lisbon,2026-Q2,9092881
